In [1]:
import os
import re
import json
import time
import requests
import pandas as pd
import numpy as np
from io import StringIO
from typing import Optional, Dict, Any, List, Tuple
from google.colab import drive
from pyliftover import LiftOver
import duckdb
from bs4 import BeautifulSoup

In [2]:
# Set this to True if using Google Colab
USE_COLAB = True

from pathlib import Path

if USE_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive")
else:
    # Change this to your local dataset directory
    DATA_ROOT = Path("/path/to/GBA1_Project")

ALPHAMISSENSE_PARQUET = DATA_ROOT / "AlphaMissense.parquet"
DBNSFP_PARQUET = DATA_ROOT / "chr1.parquet"

Mounted at /content/drive


All required paths in google drive

In [3]:
DATASET_PATH          = f"{DATA_ROOT}/Final_Dataset.csv"
OUTPUT_PATH           = f"{DATA_ROOT}/GBA1_Annotated_Final.csv"
CHECKPOINT_PATH       = f"{DATA_ROOT}/GBA1_checkpoint.csv"
ALPHAFOLD_PDB_PATH    = f"{DATA_ROOT}/GBA1_alphafold.pdb"
ALPHAMISSENSE_PARQUET = f"{DATA_ROOT}/AlphaMissense.parquet"
DBNSFP_PARQUET        = f"{DATA_ROOT}/dbNSFP_parquet/chr1.parquet"

In [ ]:
GBA1_UNIPROT_ID   = "P04062"
GBA1_DYNAMUT2_CHAIN = "A"
CHECKPOINT_INTERVAL = 10

#this is gene level so will be same for all SNPs as everything is GBA1 gene only ultimately

URLs/APIs

In [ ]:
GNOMAD_API        = "https://gnomad.broadinstitute.org/api"
INTERVAR_URL      = "http://wintervar.wglab.org/api_new.php"
ALPHAFOLD_URL     = f"https://alphafold.ebi.ac.uk/files/AF-{GBA1_UNIPROT_ID}-F1-model_v4.pdb"
DYNAMUT2_SINGLE   = "https://biosig.lab.uq.edu.au/dynamut2/api/prediction_single"
DYNAMUT2_RESULTS = "https://biosig.lab.uq.edu.au/dynamut2/results_prediction/"

In [ ]:
df = pd.read_csv(DATASET_PATH)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 759 entries, 0 to 758
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   chrom             759 non-null    int64 
 1   pos               759 non-null    int64 
 2   ref               759 non-null    object
 3   alt               759 non-null    object
 4   Consequence       759 non-null    object
 5   EXON              759 non-null    object
 6   HGVSc             759 non-null    object
 7   HGVSp             759 non-null    object
 8   cDNA_position     759 non-null    object
 9   CDS_position      759 non-null    object
 10  Protein_position  759 non-null    int64 
 11  Amino_acids       759 non-null    object
 12  Codons            759 non-null    object
 13  CLIN_SIG          254 non-null    object
dtypes: int64(3), object(11)
memory usage: 83.1+ KB


**1. AlphaMissense**

In [ ]:
alpha_con = duckdb.connect()

In [ ]:
def get_alphamissense_score(
    chrom: Any, pos: int, ref: str, alt: str
):
    """
    Looks up AlphaMissense score and class for one variant.
    Returns (am_pathogenicity, am_class) tuple, or None if not found.

    The parquet CHROM column is in 'chr1' format — we normalise chrom to match.
    """

    # Normalise chromosome to 'chr1' format
    chrom_str = str(chrom).strip().upper().replace("CHR", "")
    chrom_fmt = f"chr{chrom_str}"   # FIX: prepend 'chr' to match parquet

    try:
        result = alpha_con.execute(f"""
            SELECT pathogenicity_score, am_class
            FROM '{ALPHAMISSENSE_PARQUET}'
            WHERE CHROM = '{chrom_fmt}'
              AND POS   = {int(pos)}
              AND REF   = '{ref.strip().upper()}'
              AND ALT   = '{alt.strip().upper()}'
            LIMIT 1
        """).fetchone()

        return result if result is not None else None

    except Exception as e:
        print(f"   [WARN] AlphaMissense lookup error: {e}")
        return None

**2. gnomAD**

In [ ]:
def build_gnomad_gba1_cache_grch38():
    """
    Fetches ALL GBA1 variants from gnomAD v4 exomes (GRCh38 natively).
    Returns dict keyed by 'CHROM-POS-REF-ALT' → allele frequency.
    No liftover needed — positions match your hg38 dataset directly.

    Falls back to gnomAD v2.1 + liftover if v4 query fails.
    """

    # ── Attempt 1: gnomAD v4 GRCh38 (preferred) ─────────────────────────────
    query_v4 = """
    query {
      gene(gene_symbol: "GBA1", reference_genome: GRCh38) {
        variants(dataset: gnomad_r4_exomes) {
          chrom pos ref alt
          exome { af }
        }
      }
    }
    """
    headers = {"Content-Type": "application/json",
               "User-Agent": "GBA1-Trajectory-Pipeline/2.0"}

    print("🔄 Fetching gnomAD GBA1 cache (v4, GRCh38)...")
    try:
        resp = requests.post(GNOMAD_API, json={"query": query_v4},
                             headers=headers, timeout=60)
        resp.raise_for_status()
        data = resp.json()

        # Check for GraphQL-level errors
        if "errors" in data:
            raise ValueError(f"gnomAD GraphQL error: {data['errors']}")

        variants = (data.get("data", {})
                        .get("gene", {})
                        .get("variants", []))
        if not variants:
            raise ValueError("gnomAD v4 returned 0 variants for GBA1")

        cache = {}
        for v in variants:
            key     = f"{v['chrom']}-{v['pos']}-{v['ref']}-{v['alt']}".upper()
            exome_af = (v.get("exome") or {}).get("af", 0.0) or 0.0
            cache[key] = exome_af

        print(f"   → gnomAD v4 cache: {len(cache):,} GBA1 variants (GRCh38).")
        return cache, False   # False = no liftover needed

    except Exception as e:
        print(f"   [WARN] gnomAD v4 failed ({e}). Falling back to v2.1 + liftover...")

    # ── Fallback: gnomAD v2.1 GRCh37 + liftover ─────────────────────────────
    query_v2 = """
    query {
      gene(gene_symbol: "GBA1", reference_genome: GRCh37) {
        variants(dataset: gnomad_r2_1) {
          chrom pos ref alt
          exome  { af }
          genome { af }
        }
      }
    }
    """
    try:
        resp = requests.post(GNOMAD_API, json={"query": query_v2},
                             headers=headers, timeout=60)
        resp.raise_for_status()
        variants = (resp.json().get("data", {})
                               .get("gene", {})
                               .get("variants", []))
        cache = {}
        for v in variants:
            key      = f"{v['chrom']}-{v['pos']}-{v['ref']}-{v['alt']}".upper()
            exome_af = (v.get("exome")  or {}).get("af", 0.0) or 0.0
            gen_af   = (v.get("genome") or {}).get("af", 0.0) or 0.0
            cache[key] = exome_af if exome_af > 0 else gen_af
        print(f"   → gnomAD v2.1 fallback cache: {len(cache):,} variants (GRCh37).")
        return cache, True   # True = liftover IS needed

    except Exception as e:
        print(f"❌ Both gnomAD queries failed: {e}")
        return {}, False

In [ ]:
def get_gnomad_af_v2(
    chrom: Any, pos: int, ref: str, alt: str,
    cache: Dict[str, float],
    lo: LiftOver
):
    """Used only if fallback to v2.1 was needed (liftover path)."""
    chrom_str = str(chrom).upper().replace("CHR", "")
    converted = lo.convert_coordinate(f"chr{chrom_str}", int(pos))
    if not converted:
        return None
    _, pos_hg19, _, _ = converted[0]
    key = f"{chrom_str}-{int(pos_hg19)}-{ref.upper()}-{alt.upper()}"
    return cache.get(key)

In [ ]:
def get_gnomad_af(
    chrom: Any, pos: int, ref: str, alt: str,
    cache: Dict[str, float],
    needs_liftover: bool,
    lo: Optional[LiftOver] = None
):
    """
    Unified gnomAD lookup. If v4 cache (GRCh38): direct lookup.
    If v2.1 cache (GRCh37 fallback): liftover then lookup.
    """
    chrom_str = str(chrom).upper().replace("CHR", "")

    if needs_liftover and lo is not None:
        return get_gnomad_af_v2(chrom, pos, ref, alt, cache, lo)
    else:
        key = f"{chrom_str}-{pos}-{ref.upper()}-{alt.upper()}"
        return cache.get(key)

In [ ]:
gnomad_cache, gnomad_needs_liftover = build_gnomad_gba1_cache_grch38()
lo = LiftOver('hg38', 'hg19') if gnomad_needs_liftover else None
print(f"✅ gnomAD tool ready. "
      f"{'Using liftover (v2.1 fallback)' if gnomad_needs_liftover else 'GRCh38 native (v4)'}.")

🔄 Fetching gnomAD GBA1 cache (v4, GRCh38)...
   [WARN] gnomAD v4 failed (400 Client Error: Bad Request for url: https://gnomad.broadinstitute.org/api). Falling back to v2.1 + liftover...
   → gnomAD v2.1 fallback cache: 815 variants (GRCh37).
✅ gnomAD tool ready. Using liftover (v2.1 fallback).


**3. dbnsfp**

In [ ]:
dbnsfp_con = duckdb.connect()

3.1 PolyPhen2

In [ ]:
def get_polyphen2_data(chrom: Any, pos: int, ref: str, alt: str):
    try:
        result = dbnsfp_con.execute(f"""
        SELECT CAST(
                REGEXP_EXTRACT(
                    Polyphen2_HDIV_score,
                    '[0-9]+\\.[0-9]+'
              ) AS DOUBLE
          ) AS Polyphen2_HDIV_score,

        REGEXP_EXTRACT(
          Polyphen2_HDIV_pred,
          '[A-Za-z]+'
        ) AS Polyphen2_HDIV_pred

        FROM '{DBNSFP_PARQUET}'
        WHERE
        "#chr"='{chrom}'
        AND "pos(1-based)" = '{pos}'
        AND "ref" = '{ref}'
        AND "alt" = '{alt}'
        """).fetchone()

        return result if result is not None else None

    except Exception as e:
        print(f"   [WARN] Polyphen2 lookup error: {e}")
        return None

3.2 SIFT

In [ ]:
def get_sift_data(chrom: Any, pos: int, ref: str, alt: str):
    try:
        result = dbnsfp_con.execute(f"""
        SELECT CAST(
                REGEXP_EXTRACT(
                    SIFT_score,
                    '[0-9]+\\.[0-9]+'
              ) AS DOUBLE
          ) AS SIFT_score,

              REGEXP_EXTRACT(
                SIFT_pred,
                '[A-Za-z]+'
        ) AS SIFT_pred

        FROM '{DBNSFP_PARQUET}'
        WHERE
        "#chr"='{chrom}'
        AND "pos(1-based)" = '{pos}'
        AND "ref" = '{ref}'
        AND "alt" = '{alt}'
        """).fetchone()

        return result if result is not None else None

    except Exception as e:
        print(f"   [WARN] SIFT lookup error: {e}")
        return None

3.3 CADD

In [ ]:
def get_cadd_score(chrom: Any, pos: int, ref: str, alt: str):
  try:
    result = dbnsfp_con.execute(f"""
    SELECT CADD_raw
    FROM '{DBNSFP_PARQUET}'
    WHERE
    "#chr"='{chrom}'
    AND "pos(1-based)" = '{pos}'
    AND "ref" = '{ref}'
    AND "alt" = '{alt}'
    """).fetchone()[0]

    return result if result is not None else None

  except Exception as e:
      print(f"   [WARN] CADD lookup error: {e}")
      return None

3.4 REVEL

In [ ]:
def get_revel_score(chrom: Any, pos: int, ref: str, alt: str):
    try:
        result = dbnsfp_con.execute(f"""
        SELECT CAST(
                REGEXP_EXTRACT(
                    REVEL_score,
                    '[0-9]+\\.[0-9]+'
              ) AS DOUBLE
          ) AS REVEL_score

        FROM '{DBNSFP_PARQUET}'
        WHERE
        "#chr"='{chrom}'
        AND "pos(1-based)" = '{pos}'
        AND "ref" = '{ref}'
        AND "alt" = '{alt}'
        """).fetchone()[0]

        return result if result is not None else None

    except Exception as e:
        print(f"   [WARN] REVEL lookup error: {e}")
        return None

3.5 GERP++

In [ ]:
def get_gerp_score(chrom: Any, pos: int, ref: str, alt: str):
  try:
    result = dbnsfp_con.execute(f"""
    SELECT "GERP++_RS"
    FROM '{DBNSFP_PARQUET}'
    WHERE
    "#chr"='{chrom}'
    AND "pos(1-based)" = '{pos}'
    AND "ref" = '{ref}'
    AND "alt" = '{alt}'
    """).fetchone()[0]

    return result if result is not None else None

  except Exception as e:
      print(f"   [WARN] GERP++ lookup error: {e}")
      return None

3.6 MutPred2

In [ ]:
def get_mutpred2_data(chrom: Any, pos: int, ref: str, alt: str):
    try:
        result = dbnsfp_con.execute(f"""
        SELECT CAST(
                REGEXP_EXTRACT(
                    MutPred2_score,
                    '[0-9]+\\.[0-9]+'
              ) AS DOUBLE
          ) AS MutPred2_score,

            REGEXP_EXTRACT(
              MutPred2_pred,
              '[A-Za-z]+'
          ) AS MutPred2_pred

        FROM '{DBNSFP_PARQUET}'
        WHERE
        "#chr"='{chrom}'
        AND "pos(1-based)" = '{pos}'
        AND "ref" = '{ref}'
        AND "alt" = '{alt}'
        """).fetchone()

        return result if result is not None else None

    except Exception as e:
        print(f"   [WARN] MutPred2 lookup error: {e}")
        return None

**4. InterVar**

In [ ]:
INTERVAR_CLASS_MAP = {
    "Benign":                   0,
    "Likely benign":            1,
    "Uncertain significance":   2,
    "Likely pathogenic":        3,
    "Pathogenic":               4,
}

In [ ]:
def get_intervar(
    chrom: str, pos: int, ref: str, alt: str
):
    """
    Queries wInterVar for ACMG/AMP classification of a single missense SNV.
    Returns dict with keys:
        intervar_class  : str  — ACMG tier label
        intervar_score  : int  — ordinal 0–4 (None if no result)
        PM3_flag        : int  — 1 if PM3 evidence fires, else 0
        PVS1_flag       : int  — 1 if PVS1 fires (null variant), else 0
    Returns all-None dict on API failure.
    """
    empty = {"intervar_class": None, "intervar_score": None}

    params = {
        "queryType": "position",
        "chr":       str(chrom).replace("CHR", "").replace("chr", ""),
        "pos":       int(pos),
        "ref":       str(ref).strip().upper(),
        "alt":       str(alt).strip().upper(),
        "build":     "hg38",
    }

    for attempt in range(3):
        try:
            resp = requests.get(INTERVAR_URL, params=params, timeout=20)
            resp.raise_for_status()
            raw = resp.json()
            break
        except (requests.exceptions.Timeout,
                requests.exceptions.ConnectionError) as e:
            time.sleep(4 * (attempt + 1))
            if attempt == 2:
                print(f"   [WARN] InterVar failed after 3 attempts: {e}")
                return empty
        except Exception as e:
            print(f"   [WARN] InterVar error: {e}")
            return empty

    # Normalise response shape (API returns dict or list inconsistently)
    if not raw:
        return empty
    rows = [raw] if isinstance(raw, dict) else [r for r in raw if isinstance(r, dict)]
    if not rows:
        return empty

    # Deduplicate (GBA1/GBAP1 pseudogene can cause duplicate rows)
    seen, unique_rows = set(), []
    for row in rows:
        key = (row.get("Chromosome"), row.get("Position"),
               row.get("Ref_allele"), row.get("Alt_allele"),
               row.get("Intervar"))
        if key not in seen:
            seen.add(key)
            unique_rows.append(row)

    best = unique_rows[0]
    class_label = best.get("Intervar", "")
    score       = INTERVAR_CLASS_MAP.get(class_label)
    intervar_str= best.get("InterVar_automated", "") or ""

    return {
        "intervar_class": class_label if class_label else None,
        "intervar_score": score,
    }

**5. AlphaFold + DynaMut2**

In [ ]:
def download_alphafold_pdb(uniprot_id: str, save_path: str):
    if os.path.exists(save_path):
        print(f"   → PDB already exists at {save_path}, skipping download.")
        return True

    print(f"📥 Downloading AlphaFold PDB for {uniprot_id}...")

    # Step 1: query the AlphaFold API to get the actual current PDB URL
    # This avoids hardcoding a version number that may not exist
    try:
        api_url  = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
        api_resp = requests.get(api_url, timeout=30)
        api_resp.raise_for_status()
        entries  = api_resp.json()

        if not entries:
            print(f"❌ AlphaFold API returned no entries for {uniprot_id}")
            return False

        pdb_url = entries[0].get("pdbUrl")
        if not pdb_url:
            print(f"❌ No pdbUrl found in AlphaFold API response")
            return False

        print(f"   → Resolved PDB URL: {pdb_url}")

    except Exception as e:
        print(f"   [WARN] AlphaFold API lookup failed ({e}), trying version fallback...")

        # Step 2: fallback — try v4, v3, v2, v1 in order
        pdb_url = None
        for version in ["v4", "v3", "v2", "v1"]:
            test_url = (f"https://alphafold.ebi.ac.uk/files/"
                        f"AF-{uniprot_id}-F1-model_{version}.pdb")
            try:
                test_resp = requests.head(test_url, timeout=10)
                if test_resp.status_code == 200:
                    pdb_url = test_url
                    print(f"   → Found working URL (model {version}): {pdb_url}")
                    break
            except Exception:
                continue

        if not pdb_url:
            print(f"❌ Could not find any working AlphaFold URL for {uniprot_id}")
            return False

    # Step 3: download the PDB file from whichever URL we found
    try:
        pdb_resp = requests.get(pdb_url, timeout=60)
        pdb_resp.raise_for_status()
        with open(save_path, 'w') as f:
            f.write(pdb_resp.text)
        print(f"   → Saved to {save_path}")
        return True

    except Exception as e:
        print(f"❌ PDB download failed: {e}")
        return False

In [ ]:
def extract_plddt_per_residue(pdb_path: str):
    """
    Parses an AlphaFold PDB file and extracts pLDDT per residue.
    AlphaFold encodes pLDDT in the B-factor column of ATOM records.
    We use only CA (alpha-carbon) atoms — one per residue — to avoid
    duplicate values for the same residue.

    Returns dict {residue_number (int): plddt (float)}.
    """
    plddt_map = {}
    with open(pdb_path, 'r') as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue
            atom_name   = line[12:16].strip()
            if atom_name != "CA":
                continue
            try:
                res_num = int(line[22:26].strip())
                plddt   = float(line[60:66].strip())
                plddt_map[res_num] = plddt
            except ValueError:
                continue
    print(f"   → Extracted pLDDT for {len(plddt_map)} residues from AlphaFold PDB.")
    return plddt_map

In [ ]:
def build_mutation_string(protein_pos: Any, amino_acids: Any):
    """
    Builds DynaMut2 mutation string: {wt}{position}{mut}  e.g. 'R535P'
    Chain is passed separately to the API — do NOT include it here.
    Returns None for complex indel rows (non-numeric Protein_position).
    """
    pos_str = str(protein_pos).strip()
    if not pos_str.isdigit():
        return None
    parts = str(amino_acids).split('/') if amino_acids and '/' in str(amino_acids) else []
    if len(parts) != 2:
        return None
    wt, mut = parts[0].strip().upper(), parts[1].strip().upper()
    if len(wt) != 1 or len(mut) != 1:
        return None
    # No chain letter — DynaMut2 takes chain separately
    return f"{wt}{int(pos_str)}{mut}"

In [ ]:
# DynaMut2 polling endpoint — results are fetched here using the job_id


def get_dynamut2_ddg(mutation_str: str, pdb_path: str):
    """
    DynaMut2 is asynchronous — submits a job, then polls for result.
    Step 1: POST to prediction_single → get job_id
    Step 2: GET results/{job_id} every 5 seconds until result ready
    Returns ΔΔG in kcal/mol or None on failure.
    """
    try:
        with open(pdb_path, 'r') as f:
            pdb_content = f.read()
    except Exception as e:
        print(f"   [WARN] Could not read PDB: {e}")
        return None

    files = {"pdb_file": ("GBA1.pdb", pdb_content, "text/plain")}
    data  = {"chain": GBA1_DYNAMUT2_CHAIN, "mutation": mutation_str}

    # ── Step 1: Submit job ────────────────────────────────────────────────────
    try:
        resp = requests.post(DYNAMUT2_SINGLE, files=files,
                             data=data, timeout=30)

        resp.raise_for_status()
        submission = resp.json()

        job_id = submission.get("job_id")
        if not job_id:
            print(f"   [WARN] DynaMut2 no job_id and no ddg in response: "
                  f"{submission}")
            return None

    except Exception as e:
        print(f"   [WARN] DynaMut2 submission error for {mutation_str}: {e}")
        return None

    # ── Step 2: Poll for result ───────────────────────────────────────────────
    max_polls  = 60        # poll up to 12 times
    poll_every = 5         # seconds between each poll

    for attempt in range(max_polls):

        time.sleep(10)          # match the website refresh interval

        poll_resp = requests.get(
            f"https://biosig.lab.uq.edu.au/dynamut2/results_prediction/{job_id}",
            timeout=30
        )

        html = poll_resp.text

        # Still processing
        if "Submission status" in html or "We are processing your submission" in html:
            continue

        # Finished -> extract ddG
        soup = BeautifulSoup(html, "html.parser")
        text = soup.get_text(" ", strip=True)

        m = re.search(r'(-?\d+\.?\d*)\s*kcal/mol', text)

        if m:
            ddg = float(m.group(1))

            if ddg >= 0:
              effect = "Stabilizing"
            else:
              effect = "Destabilizing"

            return {"ddg": ddg,
                    "effect": effect}

        print("Unexpected page")
        print(text[:1000])
        return None

    print(f"[WARN] DynaMut2 job {job_id} timed out.")
    return None

    print(f"   [WARN] DynaMut2 job {job_id} timed out after "
          f"{max_polls * poll_every}s")

    return None

In [ ]:
pdb_available = download_alphafold_pdb(GBA1_UNIPROT_ID, ALPHAFOLD_PDB_PATH)

if pdb_available:
    plddt_map = extract_plddt_per_residue(ALPHAFOLD_PDB_PATH)
    with open(ALPHAFOLD_PDB_PATH, 'rb') as f:
        pdb_bytes = f.read()
    print(f"✅ AlphaFold PDB loaded ({len(pdb_bytes):,} bytes). "
          f"pLDDT map has {len(plddt_map)} residues.")
else:
    plddt_map = {}
    pdb_bytes = None
    print("⚠️  AlphaFold PDB unavailable — pLDDT and DynaMut2 will be NaN.")

   → PDB already exists at /content/drive/MyDrive/GBA1_alphafold.pdb, skipping download.
   → Extracted pLDDT for 536 residues from AlphaFold PDB.
✅ AlphaFold PDB loaded (349,028 bytes). pLDDT map has 536 residues.


Ruuning all tools on each row of the dataset and writing back into output file

In [ ]:
from pandas._libs.tslibs.offsets import CDay
def annotate_row(row: pd.Series):

    chrom = str(row['chrom'])
    pos   = int(row['pos'])
    ref   = str(row['ref']).strip().upper()
    alt   = str(row['alt']).strip().upper()
    res   = {}   # always initialised first

# ----------------- 1. AlphaMissense -----------------------------
    try:
        am = get_alphamissense_score(chrom, pos, ref, alt)
        res['AM_score'] = am[0] if am else None
        res['AM_pred']  = am[1] if am else None
    except Exception as e:
        res['AM_score'] = None
        res['AM_pred']         = None
        print(f"   [ERR] AlphaMissense row {row.name}: {e}")


# ----------------- 2. gnomAD -----------------------------
    try:
        af = get_gnomad_af(chrom, pos, ref, alt,
                           gnomad_cache, gnomad_needs_liftover, lo)
        res['gnomad_af']     = af
        res['gnomad_absent'] = 1 if af is None else 0
    except Exception as e:
        res['gnomad_af']     = None
        res['gnomad_absent'] = 1
        print(f"   [ERR] gnomAD row {row.name}: {e}")


# ----------------- 3. dbnsfp -----------------------------

# ******************* 3.1 PolyPhen2 ***********************
    try:
        pp = get_polyphen2_data(chrom, pos, ref, alt)
        res['PolyPhen2_score'] = pp[0] if pp else None
        res['PolyPhen2_pred']  = pp[1] if pp else None
    except Exception as e:
        res['PolyPhen2_score'] = None
        res['PolyPhen2_pred']  = None
        print(f"   [ERR] PolyPhen2 row {row.name}: {e}")


  # ******************* 3.2 SIFT ***********************
    try:
        st = get_sift_data(chrom, pos, ref, alt)
        res['SIFT_score'] = st[0] if st else None
        res['SIFT_pred']  = st[1] if st else None
    except Exception as e:
        res['SIFT_score'] = None
        res['SIFT_pred']  = None
        print(f"   [ERR] SIFT row {row.name}: {e}")


# ******************* 3.3 CADD ***********************
    try:
        cd = get_cadd_score(chrom, pos, ref, alt)
        res['CADD_score'] = cd if cd else None
    except Exception as e:
        res['CADD_score'] = None
        print(f"   [ERR] CADD row {row.name}: {e}")


# ******************* 3.4 REVEL ***********************
    try:
        rv = get_revel_score(chrom, pos, ref, alt)
        res['REVEL_score'] = rv if rv else None
    except Exception as e:
        res['REVEL_score'] = None
        print(f"   [ERR] REVEL row {row.name}: {e}")


# ******************* 3.5 GERP++ ***********************
    try:
        gp = get_gerp_score(chrom, pos, ref, alt)
        res['GERP++_score'] = gp if gp else None
    except Exception as e:
        res['GERP++_score'] = None
        print(f"   [ERR] GERP++ row {row.name}: {e}")


# ******************* 3.6 MutPred2 ***********************
    try:
        mp = get_mutpred2_data(chrom, pos, ref, alt)
        res['MutPred2_score'] = mp[0] if mp else None
        res['MutPred2_pred']  = mp[1] if mp else None
    except Exception as e:
        res['MutPred2_score'] = None
        res['MutPred2_pred']  = None
        print(f"   [ERR] MutPred2 row {row.name}: {e}")


# ----------------- 4. InterVar -----------------------------
    try:
        time.sleep(0.5)
        iv = get_intervar(chrom, pos, ref, alt)
        res['Intervar_class'] = iv['intervar_class']
        res['Intervar_score'] = iv['intervar_score']
    except Exception as e:
        res['Intervar_class'] = None
        res['Intervar_score'] = None
        print(f"   [ERR] InterVar row {row.name}: {e}")


# ----------------- 5. Alphafold + Dynamut2 -----------------------------
    try:
        pos_str      = str(row.get('Protein_position', '')).strip()
        prot_pos_int = int(pos_str) if pos_str.isdigit() else None
        res['plddt'] = plddt_map.get(prot_pos_int) if prot_pos_int else None
    except Exception as e:
        res['plddt'] = None
        print(f"   [ERR] pLDDT row {row.name}: {e}")
    try:
        mut_str = build_mutation_string(
            row.get('Protein_position', ''),
            row.get('Amino_acids', '')
        )
        if mut_str and os.path.exists(ALPHAFOLD_PDB_PATH):
            time.sleep(2.0)
            print(f"[Row {row.name}] DynaMut2 -> {mut_str}")
            dyn = get_dynamut2_ddg(
                mut_str,
                ALPHAFOLD_PDB_PATH
            )

            if dyn:
                res["Dynamut2_ddg"] = dyn["ddg"]
                res["Dynamut2_effect"] = dyn["effect"]
            else:
                res["Dynamut2_ddg"] = None
                res["Dynamut2_effect"] = None
        else:
            res['Dynamut2_ddg'] = None
    except Exception as e:
        res['Dynamut2_ddg'] = None
        print(f"   [ERR] DynaMut2 row {row.name}: {e}")

    return res

In [ ]:
NEW_COLS = [
    'AM_score', 'AM_pred',
    'gnomad_af', 'gnomad_absent',
    'PolyPhen2_score', 'PolyPhen2_pred',
    'SIFT_score', 'SIFT_pred',
    'CADD_score',
    'REVEL_score',
    'GERP++_score',
    'MutPred2_score', 'MutPred2_pred',
    'Intervar_class', 'Intervar_score',
    'plddt',
    'Dynamut2_ddg', 'Dynamut2_effect'
]

In [ ]:
def run_full_pipeline(
    df:              pd.DataFrame,
    output_path:     str,
    checkpoint_path: str,
):
    """
    Annotates every row of df with all tool outputs.
    Saves intermediate checkpoints every CHECKPOINT_INTERVAL rows.
    Resumes from checkpoint automatically if one exists.
    Writes final enriched CSV to output_path.
    All original columns are preserved unchanged.
    """

    # ── Resume from checkpoint if available ────────────────────────────────────
    start_idx = 0
    results   = []

    if os.path.exists(checkpoint_path):
        print(f"🔁 Checkpoint found — resuming...")
        ckpt      = pd.read_csv(checkpoint_path)
        results   = ckpt[NEW_COLS].to_dict('records')
        start_idx = len(results)
        print(f"   → Resuming from row {start_idx} / {len(df)}")
    else:
        print(f"🚀 Starting fresh — {len(df)} variants to annotate.")

    # ── Main loop ──────────────────────────────────────────────────────────────
    for i, (idx, row) in enumerate(df.iloc[start_idx:].iterrows(),
                                   start=start_idx):
        if (i + 1) % 5 == 0 or i == start_idx:
            pct = (i + 1) / len(df) * 100
            print(f"   [{i+1:>4}/{len(df)}  ({pct:>5.1f}%)]  "
                  f"{row.get('HGVSp','?')}  "
                  f"(chr{row['chrom']}:{row['pos']} {row['ref']}>{row['alt']})")

        try:
            row_result = annotate_row(row)
        except Exception as e:
            print(f"   [ERR] annotate_row crashed entirely on row {i}: {e}")
            row_result = None

# If annotate_row returned None for any reason, replace with empty dict
        if row_result is None:
            row_result = {col: None for col in NEW_COLS}

        results.append(row_result)

        # ── Checkpoint ────────────────────────────────────────────────────────
        if (i + 1) % CHECKPOINT_INTERVAL == 0:
            ckpt_df = df.iloc[:len(results)].copy()
            for col in NEW_COLS:
                ckpt_df[col] = [r.get(col) for r in results]
            ckpt_df.to_csv(checkpoint_path, index=False)
            print(f"   💾 Checkpoint saved ({len(results)} rows)")

    # ── Build final output dataframe ───────────────────────────────────────────
    print("\n✅ All rows processed. Building output CSV...")
    output_df = df.copy()   # ALL original columns preserved

    # Add per-variant tool columns
    for col in NEW_COLS:
        output_df[col] = [r.get(col) for r in results]

    output_df.to_csv(output_path, index=False)
    print(f"💾 Saved: {output_path}")
    print(f"   → Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")

    # ── Coverage report ────────────────────────────────────────────────────────
    print("\n=== COVERAGE REPORT ===")
    for col in NEW_COLS:
        n   = output_df[col].notna().sum()
        pct = n / len(output_df) * 100
        flag = "✅" if pct > 70 else ("⚠️ " if pct > 30 else "❌")
        print(f"  {flag}  {col:<35}: {n:>4}/{len(output_df)}  ({pct:.1f}%)")

    # Clean up checkpoint on successful completion
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)
        print("\n🗑️  Checkpoint removed (run complete).")

    return output_df

In [ ]:
df_annotated = run_full_pipeline(
    df              = df,
    output_path     = OUTPUT_PATH,
    checkpoint_path = CHECKPOINT_PATH,
)

print("\n Pipeline complete!")
print(f"   Output: {OUTPUT_PATH}")
print(f"   Shape:  {df_annotated.shape}")


#incase of any kind of crash, need to run this cell only and it will automatically resume from wherever it has stopped previously

🔁 Checkpoint found — resuming...
   → Resuming from row 690 / 759
   [ 691/759  ( 91.0%)]  p.Asn58Tyr  (chr1:155240021 T>A)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[Row 690] DynaMut2 -> N58Y
[Row 691] DynaMut2 -> V56G
[Row 692] DynaMut2 -> V56D
[Row 693] DynaMut2 -> V56L
   [ 695/759  ( 91.6%)]  p.Cys55Ser  (chr1:155240029 C>G)
[Row 694] DynaMut2 -> C55S
[Row 695] DynaMut2 -> V54A
[Row 696] DynaMut2 -> V54L
[Row 697] DynaMut2 -> V54M
[Row 698] DynaMut2 -> V53M
   [ 700/759  ( 92.2%)]  p.Ser52Leu  (chr1:155240038 G>A)
[Row 699] DynaMut2 -> S52L
   💾 Checkpoint saved (700 rows)
[Row 700] DynaMut2 -> S51I
[Row 701] DynaMut2 -> Y50H
[Row 702] DynaMut2 -> G49V
[Row 703] DynaMut2 -> G49D
   [ 705/759  ( 92.9%)]  p.Gly49Cys  (chr1:155240048 C>A)
[Row 704] DynaMut2 -> G49C
[Row 705] DynaMut2 -> G49S
[Row 706] DynaMut2 -> F48S
[Row 707] DynaMut2 -> S47I
[Row 708] DynaMut2 -> K46E
   [ 710/759  ( 93.5%)]  p.Pro45Leu  (chr1:155240059 G>A)
[Row 709] DynaMut2 -> P45L
   💾 Checkpoint saved (710 rows)
[Row 710] DynaMut2 -> P45S
[Row 711] DynaMut2 -> C43Y
[Row 712] DynaMut2 -> P42S
[Row 713] DynaMut2 -> P42T
   [ 715/759  ( 94.2%)]  p.Arg41Leu  (chr1:155240071 C